# SkinGraph

SkinGraph is a Python knowledge graph that surfaces representation gaps in dermatological AI datasets, with a focus on the systematic underrepresentation of darker skin tones (Fitzpatrick Scale Types IV–VI). Nodes represent datasets, skin tone types, dermatological conditions, and AI models; edges encode typed relationships such as training lineage, coverage percentages, and clinical miss rates. The goal is to make structural bias in the dermatology AI pipeline queryable and visible.

## 1. Build the Graph

Load all three seed CSVs (`datasets.csv`, `conditions.csv`, `models.csv`) and construct the `networkx.DiGraph`. The setup cell below resolves the project root and sets the working directory so relative data paths work regardless of how the notebook was launched.

In [ ]:
import os, sys, pathlib

ROOT = pathlib.Path.cwd()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))

from graph.build_graph import build

G = build()
print(G)

## 2. Query 1 — Which datasets have < 5% FST VI coverage?

`query_gap` traverses `CONTAINS` edges from dataset nodes to the `FST_VI` skin tone node and returns every dataset whose coverage fraction falls below the given threshold. A 5% threshold is used here to highlight the most severely under-represented datasets.

In [ ]:
from graph.analysis import query_gap

query_gap(G, fst='FST_VI', threshold=0.05)

## 3. Query 2 — Which models were trained on those datasets?

`models_with_gap` follows `TRAINED_ON` edges from model nodes back to the gap datasets identified above and returns the affected model names. Any model that trained on at least one under-represented dataset is flagged.

In [ ]:
from graph.analysis import models_with_gap

models_with_gap(G, fst='FST_VI', threshold=0.05)

## 4. Query 3 — Which conditions have the highest miss rates for FST IV–VI?

`condition_risk_summary` collects all condition nodes from the graph and returns them sorted by their documented `miss_rate_fst_iv_vi` in descending order. Miss rates are drawn from peer-reviewed sources cited in `data/conditions.csv`.

In [ ]:
from graph.analysis import condition_risk_summary

condition_risk_summary(G)

## 5. Visualizations

Three charts generated by `graph/visualize.py` and saved to `output/` at 150 DPI.

- **Coverage heatmap** — FST I–VI representation across all six datasets; red dashed border highlights FST V–VI.
- **Knowledge graph** — all 24 nodes and 81 edges, laid out by node type.
- **Model risk chart** — weighted FST V–VI coverage per model across its training datasets.

In [ ]:
from IPython.display import Image, display

display(Image(str(ROOT / 'output' / 'coverage_heatmap.png'), width=900))
display(Image(str(ROOT / 'output' / 'graph_viz.png'), width=900))
display(Image(str(ROOT / 'output' / 'model_risk.png'), width=900))